# Project Milestone 2

## Data Transformations

In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt

In [2]:
#read the flat file
df_orig = pd.read_csv('world-data-2023.csv')
pd.options.display.max_columns = None
df_orig.head()

,Country,Density\n(P/Km2),Abbreviation,Agricultural Land( %),Land Area(Km2),Armed Forces size,Birth Rate,Calling Code,Capital/Major City,Co2-Emissions,CPI,CPI Change (%),Currency-Code,Fertility Rate,Forested Area (%),Gasoline Price,GDP,Gross primary education enrollment (%),Gross tertiary education enrollment (%),Infant mortality,Largest city,Life expectancy,Maternal mortality ratio,Minimum wage,Official language,Out of pocket health expenditure,Physicians per thousand,Population,Population: Labor force participation (%),Tax revenue (%),Total tax rate,Unemployment rate,Urban_population,Latitude,Longitude
0,Afghanistan,60,AF,58.10%,"652,230","323,000",32.49,93.0,Kabul,"8,672",149.9,2.30%,AFN,4.47,2.10%,$0.70,"$19,101,353,833",104.00%,9.70%,47.9,Kabul,64.5,638.0,$0.43,Pashto,78.40%,0.28,"38,041,754",48.90%,9.30%,71.40%,11.12%,"9,797,273",33.939110,67.709953
1,Albania,105,AL,43.10%,"28,748","9,000",11.78,355.0,Tirana,"4,536",119.05,1.40%,ALL,1.62,28.10%,$1.36,"$15,278,077,447",107.00%,55.00%,7.8,Tirana,78.5,15.0,$1.12,Albanian,56.90%,1.20,"2,854,191",55.70%,18.60%,36.60%,12.33%,"1,747,593",41.153332,20.168331
2,Algeria,18,DZ,17.40%,"2,381,741","317,000",24.28,213.0,Algiers,"150,006",151.36,2.00%,DZD,3.02,0.80%,$0.28,"$169,988,236,398",109.90%,51.40%,20.1,Algiers,76.7,112.0,$0.95,Arabic,28.10%,1.72,"43,053,054",41.20%,37.20%,66.10%,11.70%,"31,510,100",28.033886,1.659626
3,Andorra,164,AD,40.00%,468,NaN,7.20,376.0,Andorra la Vella,469,NaN,NaN,EUR,1.27,34.00%,$1.51,"$3,154,057,987",106.40%,NaN,2.7,Andorra la Vella,NaN,NaN,$6.63,Catalan,36.40%,3.33,"77,142",NaN,NaN,NaN,NaN,"67,873",42.506285,1.521801
4,Angola,26,AO,47.50%,"1,246,700","117,000",40.73,244.0,Luanda,"34,693",261.73,17.10%,AOA,5.52,46.30%,$0.97,"$94,635,415,870",113.50%,9.30%,51.6,Luanda,60.8,241.0,$0.71,Portuguese,33.40%,0.21,"31,825,295",77.50%,9.20%,49.10%,6.89%,"21,061,025",-11.202692,17.873887


In [3]:
df_orig.dtypes

Country                                       object
Density\n(P/Km2)                              object
Abbreviation                                  object
Agricultural Land( %)                         object
Land Area(Km2)                                object
Armed Forces size                             object
Birth Rate                                   float64
Calling Code                                 float64
Capital/Major City                            object
Co2-Emissions                                 object
CPI                                           object
CPI Change (%)                                object
Currency-Code                                 object
Fertility Rate                               float64
Forested Area (%)                             object
Gasoline Price                                object
GDP                                           object
Gross primary education enrollment (%)        object
Gross tertiary education enrollment (%)       

### Step 1: clean and normalize column names

In this step, I will reformat headers by:
- removing special characters
- making names in lower case
- replacing spaces with underscores
- shortening long names

Cleaning and shortening column names makes data easier to access, reduces errors, and improves code readability. It also ensures better compatibility with analysis tools and visualizations.

In [4]:
#copy original df
df = df_orig.copy()

#clean messy column names
df.columns = (
    df.columns.str.replace('\n',' ',regex=True) #remove newline character
    .str.strip()                                #remove leading/trailing whitespaces
    .str.lower()                                #lowercase
    .str.replace(r'\s+','_',regex=True)         #replace spaces with underscores
    .str.replace(r'-','_',regex=True)           #replace hyphen with underscores
)

#rename long columns to concise names
rename = {
    'density_(p/km2)': 'density',
    'abbreviation': 'abbr',
    'agricultural_land(_%)': 'agri_pct',
    'land_area(km2)': 'land_area',
    'armed_forces_size': 'military_size',
    'capital/major_city': 'capital',
    'co2_emissions': 'co2',
    'cpi_change_(%)': 'cpi_change',
    'fertility_rate': 'fertility',
    'forested_area_(%)': 'forest_pct',
    'gasoline_price': 'gas_price',
    'gross_primary_education_enrollment_(%)': 'primary_ed',
    'gross_tertiary_education_enrollment_(%)': 'tertiary_ed',
    'infant_mortality': 'infant_mortality',
    'life_expectancy': 'life_exp',
    'maternal_mortality_ratio': 'mat_mort',
    'minimum_wage': 'min_wage',
    'official_language': 'language',
    'out_of_pocket_health_expenditure': 'oop_health',
    'physicians_per_thousand': 'physicians',
    'population:_labor_force_participation_(%)': 'labor_participation',
    'tax_revenue_(%)': 'tax_revenue_pct',
    'total_tax_rate': 'tax_rate',
    'unemployment_rate': 'unemployment',
    'urban_population': 'urban_pop'
}

df = df.rename(columns=rename)

print(df.columns.tolist())

['country', 'density', 'abbr', 'agri_pct', 'land_area', 'military_size', 'birth_rate', 'calling_code', 'capital', 'co2', 'cpi', 'cpi_change', 'currency_code', 'fertility', 'forest_pct', 'gas_price', 'gdp', 'primary_ed', 'tertiary_ed', 'infant_mortality', 'largest_city', 'life_exp', 'mat_mort', 'min_wage', 'language', 'oop_health', 'physicians', 'population', 'labor_participation', 'tax_revenue_pct', 'tax_rate', 'unemployment', 'urban_pop', 'latitude', 'longitude']


### Step 2: data type conversion

This step involves removing %, $, comma signs from the data and convert to numeric type for applicable columns

Removing them allows for numeric analysis, like plotting or aggregation.

In [5]:
#identify columns with %
percent_cols = [col for col in df.columns if df[col].astype(str).str.contains('%').any()]
print('Columns with % sign in values:', percent_cols)

#loop through the columns and remove % sign and convert to numeric
for col in percent_cols:
    df[col] = df[col].astype(str).str.replace('%','')
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.head()

Columns with % sign in values: ['agri_pct', 'cpi_change', 'forest_pct', 'primary_ed', 'tertiary_ed', 'oop_health', 'labor_participation', 'tax_revenue_pct', 'tax_rate', 'unemployment']


,country,density,abbr,agri_pct,land_area,military_size,birth_rate,calling_code,capital,co2,cpi,cpi_change,currency_code,fertility,forest_pct,gas_price,gdp,primary_ed,tertiary_ed,infant_mortality,largest_city,life_exp,mat_mort,min_wage,language,oop_health,physicians,population,labor_participation,tax_revenue_pct,tax_rate,unemployment,urban_pop,latitude,longitude
0,Afghanistan,60,AF,58.1,"652,230","323,000",32.49,93.0,Kabul,"8,672",149.9,2.3,AFN,4.47,2.1,$0.70,"$19,101,353,833",104.0,9.7,47.9,Kabul,64.5,638.0,$0.43,Pashto,78.4,0.28,"38,041,754",48.9,9.3,71.4,11.12,"9,797,273",33.939110,67.709953
1,Albania,105,AL,43.1,"28,748","9,000",11.78,355.0,Tirana,"4,536",119.05,1.4,ALL,1.62,28.1,$1.36,"$15,278,077,447",107.0,55.0,7.8,Tirana,78.5,15.0,$1.12,Albanian,56.9,1.20,"2,854,191",55.7,18.6,36.6,12.33,"1,747,593",41.153332,20.168331
2,Algeria,18,DZ,17.4,"2,381,741","317,000",24.28,213.0,Algiers,"150,006",151.36,2.0,DZD,3.02,0.8,$0.28,"$169,988,236,398",109.9,51.4,20.1,Algiers,76.7,112.0,$0.95,Arabic,28.1,1.72,"43,053,054",41.2,37.2,66.1,11.70,"31,510,100",28.033886,1.659626
3,Andorra,164,AD,40.0,468,NaN,7.20,376.0,Andorra la Vella,469,NaN,NaN,EUR,1.27,34.0,$1.51,"$3,154,057,987",106.4,NaN,2.7,Andorra la Vella,NaN,NaN,$6.63,Catalan,36.4,3.33,"77,142",NaN,NaN,NaN,NaN,"67,873",42.506285,1.521801
4,Angola,26,AO,47.5,"1,246,700","117,000",40.73,244.0,Luanda,"34,693",261.73,17.1,AOA,5.52,46.3,$0.97,"$94,635,415,870",113.5,9.3,51.6,Luanda,60.8,241.0,$0.71,Portuguese,33.4,0.21,"31,825,295",77.5,9.2,49.1,6.89,"21,061,025",-11.202692,17.873887


In [6]:
#columns with currency, commas
numeric_cols = [
    'gdp', 'gas_price', 'min_wage', 'population',
    'land_area', 'military_size', 'calling_code',
    'physicians', 'co2', 'urban_pop'
]

# clean and convert numeric columns
for col in numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(r'[\$,]','',regex=True)
    )
    df[col] = pd.to_numeric(df[col],errors='coerce')

df.head()

,country,density,abbr,agri_pct,land_area,military_size,birth_rate,calling_code,capital,co2,cpi,cpi_change,currency_code,fertility,forest_pct,gas_price,gdp,primary_ed,tertiary_ed,infant_mortality,largest_city,life_exp,mat_mort,min_wage,language,oop_health,physicians,population,labor_participation,tax_revenue_pct,tax_rate,unemployment,urban_pop,latitude,longitude
0,Afghanistan,60,AF,58.1,652230.0,323000.0,32.49,93.0,Kabul,8672.0,149.9,2.3,AFN,4.47,2.1,0.70,1.910135e+10,104.0,9.7,47.9,Kabul,64.5,638.0,0.43,Pashto,78.4,0.28,38041754.0,48.9,9.3,71.4,11.12,9797273.0,33.939110,67.709953
1,Albania,105,AL,43.1,28748.0,9000.0,11.78,355.0,Tirana,4536.0,119.05,1.4,ALL,1.62,28.1,1.36,1.527808e+10,107.0,55.0,7.8,Tirana,78.5,15.0,1.12,Albanian,56.9,1.20,2854191.0,55.7,18.6,36.6,12.33,1747593.0,41.153332,20.168331
2,Algeria,18,DZ,17.4,2381741.0,317000.0,24.28,213.0,Algiers,150006.0,151.36,2.0,DZD,3.02,0.8,0.28,1.699882e+11,109.9,51.4,20.1,Algiers,76.7,112.0,0.95,Arabic,28.1,1.72,43053054.0,41.2,37.2,66.1,11.70,31510100.0,28.033886,1.659626
3,Andorra,164,AD,40.0,468.0,NaN,7.20,376.0,Andorra la Vella,469.0,NaN,NaN,EUR,1.27,34.0,1.51,3.154058e+09,106.4,NaN,2.7,Andorra la Vella,NaN,NaN,6.63,Catalan,36.4,3.33,77142.0,NaN,NaN,NaN,NaN,67873.0,42.506285,1.521801
4,Angola,26,AO,47.5,1246700.0,117000.0,40.73,244.0,Luanda,34693.0,261.73,17.1,AOA,5.52,46.3,0.97,9.463542e+10,113.5,9.3,51.6,Luanda,60.8,241.0,0.71,Portuguese,33.4,0.21,31825295.0,77.5,9.2,49.1,6.89,21061025.0,-11.202692,17.873887


Further analysis revealed that some fields included values '�'.

In [7]:
#remove � character from country, capital, and largest_city columns
cols = ['country','capital','largest_city']

for col in cols:
    df[col] = df[col].astype(str).str.replace('�','')

### Step 3: Missing values

In this step, I will handle missing values by filling them with median values (numeric) or 'Unknown' (string).
This process is necessary because null values can cause errors in analysis or modeling. Using the median avoids distortion from outliers and using 'Unkown' keeps categorical columns usable. However, this may introduce biases.

In [8]:
#check nulls in all columns
print(df.isna().any())

#check percentage of missing values in each column
(df.isna().mean() * 100)

country                False
density                False
abbr                    True
agri_pct                True
land_area               True
military_size           True
birth_rate              True
calling_code            True
capital                False
co2                     True
cpi                     True
cpi_change              True
currency_code           True
fertility               True
forest_pct              True
gas_price               True
gdp                     True
primary_ed              True
tertiary_ed             True
infant_mortality        True
largest_city           False
life_exp                True
mat_mort                True
min_wage                True
language                True
oop_health              True
physicians              True
population              True
labor_participation     True
tax_revenue_pct         True
tax_rate                True
unemployment            True
urban_pop               True
latitude                True
longitude     

country                 0.000000
density                 0.000000
abbr                    3.589744
agri_pct                3.589744
land_area               0.512821
military_size          12.307692
birth_rate              3.076923
calling_code            0.512821
capital                 0.000000
co2                     3.589744
cpi                     8.717949
cpi_change              8.205128
currency_code           7.692308
fertility               3.589744
forest_pct              3.589744
gas_price              10.256410
gdp                     1.025641
primary_ed              3.589744
tertiary_ed             6.153846
infant_mortality        3.076923
largest_city            0.000000
life_exp                4.102564
mat_mort                7.179487
min_wage               23.076923
language                2.564103
oop_health              3.589744
physicians              3.589744
population              0.512821
labor_participation     9.743590
tax_revenue_pct        13.333333
tax_rate  

In [9]:
# fill numeric columns with median values
for col in df.select_dtypes(include='number'):
    df[col] = df[col].fillna(df[col].median())

#fill string columns with Unknown
for col in df.select_dtypes(include='object'):
    df[col] = df[col].fillna('Unkown')

print(df.isna().sum().any())

False


### Step 4: identify outliers

This step involves identifying outliers or bad data using z-score for selected numeric columns, such as GDP or population. Values with a z-score greater than 3 or less than -3 are considered statistically distant from the mean and can be flagged as outliers. Identifying these helps prevent extreme values from influencing analysis.

In [10]:
from scipy.stats import zscore

#numeric columns to identify
columns_to_check = ['gdp', 'population', 'fertility', 'birth_rate']

#calculate z-scores and check if the absolute value is greater than 3 
z_scores = df[columns_to_check].apply(zscore)
outlier_check = (z_scores.abs() > 3).any(axis=1)

# retrieve outlier rows
outliers = df[outlier_check]

outliers.head()

,country,density,abbr,agri_pct,land_area,military_size,birth_rate,calling_code,capital,co2,cpi,cpi_change,currency_code,fertility,forest_pct,gas_price,gdp,primary_ed,tertiary_ed,infant_mortality,largest_city,life_exp,mat_mort,min_wage,language,oop_health,physicians,population,labor_participation,tax_revenue_pct,tax_rate,unemployment,urban_pop,latitude,longitude
36,China,153,CN,56.2,9596960.0,2695000.0,10.90,86.0,Beijing,9893038.0,125.08,2.9,CNY,1.69,22.4,0.96,1.991000e+13,100.2,50.6,7.4,Shanghai,77.0,29.0,0.87,Standard Chinese,32.4,1.98,1.397715e+09,68.0,9.4,59.2,4.32,842933962.0,35.861660,104.195397
77,India,464,IN,60.4,3287263.0,3031000.0,17.86,91.0,New Delhi,2407672.0,180.44,7.7,INR,2.22,23.8,0.97,2.611000e+12,113.0,28.1,29.9,Kurebhar,69.4,145.0,0.30,Hindi,65.1,0.86,1.366418e+09,49.3,11.2,49.7,5.36,471031528.0,20.593684,78.962880
125,Niger,19,NE,36.1,1267000.0,10000.0,46.08,227.0,Niamey,2017.0,109.32,-2.5,XOF,6.91,0.9,0.88,1.292815e+10,74.7,4.4,48.0,Niamey,62.0,509.0,0.29,French,52.3,0.04,2.331072e+07,72.0,11.8,47.2,0.47,3850231.0,17.607789,8.081666
186,United States,36,US,44.4,9833517.0,1359000.0,11.60,1.0,"Washington, D.C.",5006302.0,117.24,7.5,USD,1.73,33.9,0.71,2.142770e+13,101.8,88.2,5.6,New York City,78.5,19.0,7.25,Unkown,11.1,2.61,3.282395e+08,62.0,9.6,36.6,14.70,270663028.0,37.090240,-95.712891


### Step 5: Duplicates

This step identifies exact duplicate rows in the dataset. This ensures that records are not overrepresented in analysis.

In [11]:

#Check for fully duplicated rows
duplicates = df[df.duplicated()]

print(duplicates.shape[0])

#check for duplicates within country name
df[df.duplicated(subset=['country'])]

0


,country,density,abbr,agri_pct,land_area,military_size,birth_rate,calling_code,capital,co2,cpi,cpi_change,currency_code,fertility,forest_pct,gas_price,gdp,primary_ed,tertiary_ed,infant_mortality,largest_city,life_exp,mat_mort,min_wage,language,oop_health,physicians,population,labor_participation,tax_revenue_pct,tax_rate,unemployment,urban_pop,latitude,longitude


### Step 6: Derived columns

This step creates new columns that represent ratios or percentages dervied from existing values, such as GDP per capita or military size as a percent of the population. These transformations help uncover patterns and enable fair comparisons between countries of different sizes

In [12]:
#calculate and create new columns
df['gdp_per_capita'] = df['gdp'] / df['population']
df['military_pct'] = (df['military_size'] / df['population']) * 100
df['urban_pct'] = (df['urban_pop'] / df['population']) * 100

df.head()

,country,density,abbr,agri_pct,land_area,military_size,birth_rate,calling_code,capital,co2,cpi,cpi_change,currency_code,fertility,forest_pct,gas_price,gdp,primary_ed,tertiary_ed,infant_mortality,largest_city,life_exp,mat_mort,min_wage,language,oop_health,physicians,population,labor_participation,tax_revenue_pct,tax_rate,unemployment,urban_pop,latitude,longitude,gdp_per_capita,military_pct,urban_pct
0,Afghanistan,60,AF,58.1,652230.0,323000.0,32.49,93.0,Kabul,8672.0,149.9,2.3,AFN,4.47,2.1,0.70,1.910135e+10,104.0,9.7,47.9,Kabul,64.5,638.0,0.43,Pashto,78.4,0.28,38041754.0,48.90,9.3,71.4,11.12,9797273.0,33.939110,67.709953,502.115487,0.849067,25.753999
1,Albania,105,AL,43.1,28748.0,9000.0,11.78,355.0,Tirana,4536.0,119.05,1.4,ALL,1.62,28.1,1.36,1.527808e+10,107.0,55.0,7.8,Tirana,78.5,15.0,1.12,Albanian,56.9,1.20,2854191.0,55.70,18.6,36.6,12.33,1747593.0,41.153332,20.168331,5352.857411,0.315326,61.229014
2,Algeria,18,DZ,17.4,2381741.0,317000.0,24.28,213.0,Algiers,150006.0,151.36,2.0,DZD,3.02,0.8,0.28,1.699882e+11,109.9,51.4,20.1,Algiers,76.7,112.0,0.95,Arabic,28.1,1.72,43053054.0,41.20,37.2,66.1,11.70,31510100.0,28.033886,1.659626,3948.343279,0.736301,73.189001
3,Andorra,164,AD,40.0,468.0,31000.0,7.20,376.0,Andorra la Vella,469.0,Unkown,2.3,EUR,1.27,34.0,1.51,3.154058e+09,106.4,31.2,2.7,Andorra la Vella,73.2,53.0,6.63,Catalan,36.4,3.33,77142.0,62.45,16.3,37.2,5.36,67873.0,42.506285,1.521801,40886.391162,40.185632,87.984496
4,Angola,26,AO,47.5,1246700.0,117000.0,40.73,244.0,Luanda,34693.0,261.73,17.1,AOA,5.52,46.3,0.97,9.463542e+10,113.5,9.3,51.6,Luanda,60.8,241.0,0.71,Portuguese,33.4,0.21,31825295.0,77.50,9.2,49.1,6.89,21061025.0,-11.202692,17.873887,2973.591160,0.367632,66.176999


### Final output

In [13]:
df.head(25)

,country,density,abbr,agri_pct,land_area,military_size,birth_rate,calling_code,capital,co2,cpi,cpi_change,currency_code,fertility,forest_pct,gas_price,gdp,primary_ed,tertiary_ed,infant_mortality,largest_city,life_exp,mat_mort,min_wage,language,oop_health,physicians,population,labor_participation,tax_revenue_pct,tax_rate,unemployment,urban_pop,latitude,longitude,gdp_per_capita,military_pct,urban_pct
0,Afghanistan,60,AF,58.1,652230.0,323000.0,32.49,93.0,Kabul,8672.0,149.9,2.3,AFN,4.47,2.1,0.70,1.910135e+10,104.00,9.7,47.9,Kabul,64.5,638.0,0.430,Pashto,78.4,0.28,38041754.0,48.90,9.3,71.4,11.12,9797273.0,33.939110,67.709953,502.115487,0.849067,25.753999
1,Albania,105,AL,43.1,28748.0,9000.0,11.78,355.0,Tirana,4536.0,119.05,1.4,ALL,1.62,28.1,1.36,1.527808e+10,107.00,55.0,7.8,Tirana,78.5,15.0,1.120,Albanian,56.9,1.20,2854191.0,55.70,18.6,36.6,12.33,1747593.0,41.153332,20.168331,5352.857411,0.315326,61.229014
2,Algeria,18,DZ,17.4,2381741.0,317000.0,24.28,213.0,Algiers,150006.0,151.36,2.0,DZD,3.02,0.8,0.28,1.699882e+11,109.90,51.4,20.1,Algiers,76.7,112.0,0.950,Arabic,28.1,1.72,43053054.0,41.20,37.2,66.1,11.70,31510100.0,28.033886,1.659626,3948.343279,0.736301,73.189001
3,Andorra,164,AD,40.0,468.0,31000.0,7.20,376.0,Andorra la Vella,469.0,Unkown,2.3,EUR,1.27,34.0,1.51,3.154058e+09,106.40,31.2,2.7,Andorra la Vella,73.2,53.0,6.630,Catalan,36.4,3.33,77142.0,62.45,16.3,37.2,5.36,67873.0,42.506285,1.521801,40886.391162,40.185632,87.984496
4,Angola,26,AO,47.5,1246700.0,117000.0,40.73,244.0,Luanda,34693.0,261.73,17.1,AOA,5.52,46.3,0.97,9.463542e+10,113.50,9.3,51.6,Luanda,60.8,241.0,0.710,Portuguese,33.4,0.21,31825295.0,77.50,9.2,49.1,6.89,21061025.0,-11.202692,17.873887,2973.591160,0.367632,66.176999
5,Antigua and Barbuda,223,AG,20.5,443.0,0.0,15.33,1.0,"St. John's, Saint John",557.0,113.81,1.2,XCD,1.99,22.3,0.99,1.727759e+09,105.00,24.8,5.0,"St. John's, Saint John",76.9,42.0,3.040,English,24.3,2.76,97118.0,62.45,16.5,43.0,5.36,23800.0,17.060816,-61.796428,17790.309304,0.000000,24.506271
6,Argentina,17,AR,54.3,2780400.0,105000.0,17.02,54.0,Buenos Aires,201348.0,232.75,53.5,ARS,2.26,9.8,1.10,4.496634e+11,109.70,90.0,8.8,Buenos Aires,76.5,39.0,3.350,Spanish,17.6,3.96,44938712.0,61.30,10.1,106.3,9.79,41339571.0,-38.416097,-63.616672,10006.148974,0.233652,91.991001
7,Armenia,104,AM,58.9,29743.0,49000.0,13.99,374.0,Yerevan,5156.0,129.18,1.4,AMD,1.76,11.7,0.77,1.367280e+10,92.70,54.6,11.0,Yerevan,74.9,26.0,0.660,Armenian,81.6,4.40,2957731.0,55.60,20.9,22.6,16.99,1869848.0,40.069099,45.038189,4622.733493,1.656675,63.219001
8,Australia,3,AU,48.2,7741220.0,58000.0,12.60,61.0,Canberra,375908.0,119.8,1.6,AUD,1.74,16.3,0.93,1.392681e+12,100.30,113.1,3.1,Sydney,82.7,6.0,13.590,Unkown,19.6,3.68,25766605.0,65.50,23.0,47.4,5.27,21844756.0,-25.274398,133.775136,54049.828812,0.225098,84.779334
9,Austria,109,AT,32.4,83871.0,21000.0,9.70,43.0,Vienna,61448.0,118.06,1.5,EUR,1.47,46.9,1.20,4.463147e+11,103.10,85.1,2.9,Vienna,81.6,5.0,1.045,German,17.9,5.17,8877067.0,60.70,25.4,51.4,4.67,5194416.0,47.516231,14.550072,50277.275087,0.236565,58.515003


In [15]:
df['country'] = df["country"].str.lower()
df.to_csv("flat_file.csv",index=False)